# Does our saved data justify a 24-hour receipt overlap?

**Offline diagnostic only.** No API calls, Snowflake operations, or changes to pipeline settings.

Start with the approved frozen baseline and follow its saved child snapshots. Compare each parent's complete report arrays with the child's original extraction pages. A newly observed full payload with `received_at` **before the parent cutoff** would be missed by a hypothetical extraction starting exactly at that cutoff.

This tests observed overlap recovery, not drive-end-to-receipt delay. It cannot prove the API's availability guarantees or an optimal lookback. The saved child requests already used 24 hours, so older missed records are unobservable here. Unseen data cannot be recovered by simulating a 36-hour filter on a 24-hour extract.


In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timedelta
import hashlib
import json
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'dbt/dbt_project.yml').is_file())
BASELINE = 'e943b3aefc3411575c251ada7404c7013b54bf9510e8d37e9e69bd2f80bf863a'
SNAPSHOTS = ROOT / 'data/snapshots'

def canonical(record):
    return json.dumps(record, sort_keys=True, separators=(',', ':'),
                      ensure_ascii=False, allow_nan=False)

def receipt(record):
    value = record.get('received_at')
    return datetime.fromisoformat(value.replace('Z', '+00:00')) if value else None

def read_checked(folder, manifest, relative):
    # Verify only the source files this diagnostic reads; do not modify them.
    path = folder / relative
    expected = manifest['identity']['source_files'][relative]
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != expected:
        raise ValueError('Source checksum mismatch: ' + relative)
    return json.loads(path.read_text())

manifests = {}
for path in SNAPSHOTS.glob('*/manifest.json'):
    manifest = json.loads(path.read_text())
    if manifest['identity']['purpose'] == 'FULL_BASELINE':
        manifests[manifest['snapshot_id']] = manifest
assert BASELINE in manifests, 'Original baseline files are required.'
print('Available complete local snapshots:', len(manifests))


## Compare each saved extraction with its actual parent

The baseline has no original drive/warning response pages. We can still use its complete saved arrays as the parent of the first fresh extraction. Later snapshots preserve their downloaded pages.

Count full-payload occurrences rather than distinct session IDs, matching `merge_overlap()`. Compare against **all** parent records, including receipts after its analytical cutoff: those were already captured. Assume an inclusive lower-bound filter for the hypothetical no-overlap request; exact-boundary cases are reported separately.


In [ ]:
# Follow direct descendants of the approved baseline in cutoff order.
reachable = {BASELINE}
pairs = []
for child_id, child in sorted(manifests.items(), key=lambda item:
                             item[1]['identity']['analysis_receipt_cutoff_utc']):
    parent_id = child['identity'].get('parent_snapshot_id')
    if parent_id in reachable:
        pairs.append((parent_id, child_id))
        reachable.add(child_id)
assert pairs, 'No saved child extraction follows the baseline.'

rows = []
for parent_id, child_id in pairs:
    parent, child = manifests[parent_id], manifests[child_id]
    cutoff = datetime.fromisoformat(parent['identity']['analysis_receipt_cutoff_utc'])
    child_cutoff = datetime.fromisoformat(child['identity']['analysis_receipt_cutoff_utc'])
    for endpoint in ('drives', 'warnings'):
        parent_records = read_checked(SNAPSHOTS / parent_id, parent,
                                     parent['endpoints'][endpoint]['source_file'])['records']
        prior = Counter(canonical(record) for record in parent_records)
        incoming = Counter()
        fetched = new = recovered = boundary = future = invalid = 0
        gaps = []
        prefix = f'sources/extractions/{endpoint}/'
        page_paths = sorted(name for name in child['identity']['source_files']
                            if name.startswith(prefix + 'page_') and name.endswith('.json'))
        assert page_paths, 'Original extraction pages missing.'
        for page_path in page_paths:
            page = read_checked(SNAPSHOTS / child_id, child, page_path)
            for record in page['data']:
                fetched += 1
                key = canonical(record)
                incoming[key] += 1
                if incoming[key] <= prior[key]:
                    continue  # Already represented by parent multiplicity.
                new += 1
                timestamp = receipt(record)
                if timestamp is None:
                    invalid += 1
                elif timestamp > child_cutoff:
                    future += 1  # Preserved in RAW but outside this run's analysis.
                elif timestamp < cutoff:
                    recovered += 1
                    gaps.append((cutoff - timestamp).total_seconds() / 3600)
                elif timestamp == cutoff:
                    boundary += 1
        requested_since = child['identity']['capture'][endpoint]['received_since']
        assert datetime.fromisoformat(requested_since) == cutoff - timedelta(hours=24)
        rows.append({
            'child_snapshot': child_id[:12], 'endpoint': endpoint,
            'parent_cutoff_utc': cutoff.isoformat(),
            'fetched_reports': fetched, 'new_report_occurrences': new,
            'additional_pre_cutoff_reports': recovered,
            'new_exact_cutoff_reports': boundary,
            'new_after_child_cutoff': future, 'new_missing_receipt': invalid,
            'largest_observed_recovery_hours': max(gaps) if gaps else None
        })
results = pd.DataFrame(rows)
display(results)


## What can we conclude?

- A positive additional-pre-cutoff count supports **some overlap** for those observations; it does not establish that 24 hours is optimal.
- Zero means no additional pre-cutoff report occurrences were recovered in these saved comparisons. It does not establish that no overlap is safe forever.
- New payload occurrences can reflect revised reports, duplicate multiplicity changes, or source/pagination behavior. They do not prove the exact first API-visible time.
- Checkpoint timestamps record when a page was fetched, not when its reports first became available.
- This cannot identify reports outside the saved 24-hour request window, source deletions, or a transactional API snapshot.


In [ ]:
total_recovered = int(results['additional_pre_cutoff_reports'].sum())
print('Additional pre-cutoff report occurrences recovered:', total_recovered)
if total_recovered:
    print('Some overlap helped in the saved comparisons. The evidence does not justify 24 hours specifically.')
else:
    print('The saved comparisons provide no observed recovery benefit from the overlap.')
    print('24 hours remains an unvalidated precaution; zero observed benefit is not proof that removal is safe.')
print('A 36-hour alternative cannot be evaluated from these 24-hour extracts.')
print('Pipeline logic has not been changed.')
